In [1]:
from pathlib import Path
import os

print("Current working directory:", os.getcwd())
print("D drive exists:", Path("D:/").exists())
print("Can write here:", os.access(os.getcwd(), os.W_OK))

Current working directory: D:\GPAC
D drive exists: True
Can write here: True


In [2]:
from pathlib import Path

PROJECT_DIR = Path(r"D:\GPAC")

for item in PROJECT_DIR.iterdir():
    if item.is_file():
        size_mb = item.stat().st_size / (1024**2)
        print(f"FILE   {item.name} — {size_mb:.1f} MB")
    else:
        print(f"FOLDER {item.name}")

FOLDER .ipynb_checkpoints
FILE   GPAC.txt — 1013.8 MB
FILE   Untitled.ipynb — 0.0 MB


In [3]:
CORPUS_PATH = PROJECT_DIR / "GPAC.txt"

print("Corpus exists:", CORPUS_PATH.is_file())

if CORPUS_PATH.is_file():
    print(f"Corpus size: {CORPUS_PATH.stat().st_size / (1024**3):.3f} GiB")

Corpus exists: True
Corpus size: 0.990 GiB


In [4]:

from pathlib import Path
from collections import Counter
import re
import time

CORPUS_PATH = Path(r"D:\GPAC\GPAC.txt")

SENTENCE_END_RE = re.compile(r"፡፡|።|፧|\?|!+")
PUNCTUATION_RE = re.compile(
    r"""[፡።፣፤፥፦፧፨?!,.;:()\[\]{}<>"'/%—–-]+"""
)

SAMPLE_SIZE = 5_000_000

start = time.perf_counter()

with CORPUS_PATH.open("r", encoding="utf-8") as f:
    sample = f.read(SAMPLE_SIZE)

# Count only complete sentences, avoiding a partial final sentence.
matches = list(SENTENCE_END_RE.finditer(sample))
if not matches:
    raise ValueError("No sentence-ending punctuation found in sample.")

sample = sample[:matches[-1].end()]
raw_sentences = SENTENCE_END_RE.split(sample)

unigrams = Counter()
bigrams = Counter()
sentence_count = 0
token_count = 0

for sentence in raw_sentences:
    cleaned = PUNCTUATION_RE.sub(" ", sentence)
    tokens = cleaned.split()

    if not tokens:
        continue

    sentence_count += 1
    token_count += len(tokens)
    unigrams.update(tokens)
    bigrams.update(zip(tokens, tokens[1:]))

elapsed = time.perf_counter() - start

print(f"Characters read: {len(sample):,}")
print(f"Sentences counted: {sentence_count:,}")
print(f"Tokens counted: {token_count:,}")
print(f"Unique unigrams: {len(unigrams):,}")
print(f"Unique bigrams: {len(bigrams):,}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print(f"Estimated full-file time at this rate: "
      f"{elapsed * CORPUS_PATH.stat().st_size / len(sample) / 3600:.2f} hours")

print("\nTop 10 unigrams:")
for word, count in unigrams.most_common(10):
    print(repr(word), count)

print("\nTop 10 bigrams:")
for pair, count in bigrams.most_common(10):
    print(" ".join(pair), count)


Characters read: 4,999,970
Sentences counted: 75,784
Tokens counted: 945,371
Unique unigrams: 166,363
Unique bigrams: 668,745
Elapsed time: 3.11 seconds
Estimated full-file time at this rate: 0.18 hours

Top 10 unigrams:
'ነው' 17420
'ላይ' 8180
'ግን' 4708
'ወደ' 4522
'ነበር' 4327
'ውስጥ' 4133
'እና' 3891
'ጋር' 3586
'ነገር' 3379
'አንድ' 3147

Top 10 bigrams:
ዓ ም 1031
ነገር ግን 668
ማለት ነው 564
ብቻ ሳይሆን 489
ብቻ ነው 469
ዶ ር 350
አዲስ አበባ 331
ሚሊዮን ዶላር 309
ምን ያህል 278
ወ ሮ 214


In [5]:
from pathlib import Path
import shutil

PROJECT_DIR = Path(r"D:\GPAC")
CORPUS_PATH = PROJECT_DIR / "GPAC.txt"
DB_PATH = PROJECT_DIR / "ngram_counts_local.sqlite"

disk = shutil.disk_usage(PROJECT_DIR)

print(f"Free space on D: {disk.free / (1024**3):.2f} GiB")
print(f"Corpus size: {CORPUS_PATH.stat().st_size / (1024**3):.3f} GiB")
print(f"New database path: {DB_PATH}")
print(f"Database already exists: {DB_PATH.exists()}")

print("\nExisting database-related files:")
for p in PROJECT_DIR.glob("*.sqlite*"):
    print(f"{p.name} — {p.stat().st_size / (1024**3):.3f} GiB")

Free space on D: 164.86 GiB
Corpus size: 0.990 GiB
New database path: D:\GPAC\ngram_counts_local.sqlite
Database already exists: False

Existing database-related files:


In [6]:

from pathlib import Path
from collections import Counter
import codecs
import re
import sqlite3
import time

# --------------------------------------------------
# 1. Paths and settings
# --------------------------------------------------
PROJECT_DIR = Path(r"D:\GPAC")
CORPUS_PATH = PROJECT_DIR / "GPAC.txt"
DB_PATH = PROJECT_DIR / "ngram_counts_local.sqlite"

CHUNK_BYTES = 1_000_000
COMMIT_EVERY_CHUNKS = 5
PROGRESS_EVERY_BATCHES = 10

SENTENCE_END_RE = re.compile(r"፡፡|።|፧|\?|!+")
PUNCTUATION_RE = re.compile(
    r"""[፡።፣፤፥፦፧፨?!,.;:()\[\]{}<>"'/%—–-]+"""
)

# --------------------------------------------------
# 2. Validate paths
# --------------------------------------------------
if not CORPUS_PATH.is_file():
    raise FileNotFoundError(f"Corpus not found: {CORPUS_PATH}")

if DB_PATH.exists():
    raise FileExistsError(
        f"Database already exists: {DB_PATH}\n"
        "This script will not overwrite it. Inspect or resume it instead."
    )

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
corpus_size = CORPUS_PATH.stat().st_size

# --------------------------------------------------
# 3. Create database and checkpoint
# --------------------------------------------------
conn = sqlite3.connect(DB_PATH, timeout=60)

conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA synchronous=NORMAL")
conn.execute("PRAGMA cache_size=-64000")

conn.executescript("""
CREATE TABLE unigrams (
    token TEXT PRIMARY KEY,
    count INTEGER NOT NULL
);

CREATE TABLE bigrams (
    w1 TEXT NOT NULL,
    w2 TEXT NOT NULL,
    count INTEGER NOT NULL,
    PRIMARY KEY (w1, w2)
);

CREATE TABLE run_state (
    id INTEGER PRIMARY KEY CHECK (id = 1),
    corpus_path TEXT NOT NULL,
    corpus_size INTEGER NOT NULL,
    byte_offset INTEGER NOT NULL,
    carry TEXT NOT NULL,
    sentences INTEGER NOT NULL,
    tokens INTEGER NOT NULL,
    complete INTEGER NOT NULL
);
""")

conn.execute("""
INSERT INTO run_state
VALUES (1, ?, ?, 0, '', 0, 0, 0)
""", (str(CORPUS_PATH.resolve()), corpus_size))
conn.commit()

# --------------------------------------------------
# 4. Counting helpers
# --------------------------------------------------
batch_unigrams = Counter()
batch_bigrams = Counter()
batch_sentences = 0
batch_tokens = 0

def count_sentence(sentence):
    global batch_sentences, batch_tokens

    cleaned = PUNCTUATION_RE.sub(" ", sentence)
    tokens = cleaned.split()

    if not tokens:
        return

    batch_unigrams.update(tokens)
    batch_bigrams.update(zip(tokens, tokens[1:]))

    batch_sentences += 1
    batch_tokens += len(tokens)


def commit_batch(offset, carry, total_sentences,
                 total_tokens, complete=False):
    """Save counts and their matching checkpoint atomically."""
    global batch_sentences, batch_tokens

    with conn:
        conn.executemany("""
            INSERT INTO unigrams(token, count)
            VALUES (?, ?)
            ON CONFLICT(token) DO UPDATE SET
                count = count + excluded.count
        """, batch_unigrams.items())

        conn.executemany("""
            INSERT INTO bigrams(w1, w2, count)
            VALUES (?, ?, ?)
            ON CONFLICT(w1, w2) DO UPDATE SET
                count = count + excluded.count
        """, [
            (w1, w2, count)
            for (w1, w2), count in batch_bigrams.items()
        ])

        conn.execute("""
            UPDATE run_state
            SET byte_offset = ?,
                carry = ?,
                sentences = ?,
                tokens = ?,
                complete = ?
            WHERE id = 1
        """, (
            offset, carry,
            total_sentences + batch_sentences,
            total_tokens + batch_tokens,
            int(complete)
        ))

    total_sentences += batch_sentences
    total_tokens += batch_tokens

    batch_unigrams.clear()
    batch_bigrams.clear()
    batch_sentences = 0
    batch_tokens = 0

    return total_sentences, total_tokens


# --------------------------------------------------
# 5. Stream through the corpus
# --------------------------------------------------
start_time = time.perf_counter()
total_sentences = 0
total_tokens = 0
batches = 0
chunks_since_commit = 0

carry = ""
decoder = codecs.getincrementaldecoder("utf-8")()

try:
    with CORPUS_PATH.open("rb") as f:
        while True:
            raw = f.read(CHUNK_BYTES)

            if not raw:
                # Flush the UTF-8 decoder at EOF.
                carry += decoder.decode(b"", final=True)

                # Process all complete sentences remaining.
                last_end = 0
                for match in SENTENCE_END_RE.finditer(carry):
                    count_sentence(carry[last_end:match.end()])
                    last_end = match.end()

                # Count a final sentence even without ending punctuation.
                count_sentence(carry[last_end:])

                total_sentences, total_tokens = commit_batch(
                    f.tell(), "", total_sentences,
                    total_tokens, complete=True
                )
                break

            carry += decoder.decode(raw, final=False)

            # Only count complete sentences. Keep the unfinished
            # sentence in carry for the next chunk.
            last_end = 0
            for match in SENTENCE_END_RE.finditer(carry):
                count_sentence(carry[last_end:match.end()])
                last_end = match.end()

            carry = carry[last_end:]

            # Rewind the checkpoint over any incomplete UTF-8 bytes.
            pending_bytes = decoder.getstate()[0]
            safe_offset = f.tell() - len(pending_bytes)

            chunks_since_commit += 1

            if chunks_since_commit >= COMMIT_EVERY_CHUNKS:
                total_sentences, total_tokens = commit_batch(
                    safe_offset, carry, total_sentences,
                    total_tokens
                )

                chunks_since_commit = 0
                batches += 1

                if batches % PROGRESS_EVERY_BATCHES == 0:
                    elapsed = time.perf_counter() - start_time
                    percent = 100 * safe_offset / corpus_size

                    print(
                        f"{percent:.1f}% | "
                        f"{safe_offset:,}/{corpus_size:,} bytes | "
                        f"Sentences: {total_sentences:,} | "
                        f"Tokens: {total_tokens:,} | "
                        f"Elapsed: {elapsed / 60:.1f} min",
                        flush=True
                    )

    elapsed = time.perf_counter() - start_time
    print("\nCounting finished.")
    print(f"Sentences: {total_sentences:,}")
    print(f"Tokens: {total_tokens:,}")
    print(f"Elapsed: {elapsed / 60:.2f} minutes")
    print(f"Database: {DB_PATH}")

except Exception:
    print(
        "\nThe run stopped. Previously committed batches and their "
        "checkpoint remain in the database. Do not delete the database."
    )
    raise

finally:
    conn.close()










4.7% | 50,000,000/1,063,098,071 bytes | Sentences: 291,831 | Tokens: 3,697,055 | Elapsed: 4.0 min
9.4% | 99,999,998/1,063,098,071 bytes | Sentences: 588,703 | Tokens: 7,374,387 | Elapsed: 8.9 min
14.1% | 149,999,998/1,063,098,071 bytes | Sentences: 844,512 | Tokens: 11,013,483 | Elapsed: 16.8 min
18.8% | 200,000,000/1,063,098,071 bytes | Sentences: 1,086,307 | Tokens: 14,643,444 | Elapsed: 28.5 min
23.5% | 249,999,998/1,063,098,071 bytes | Sentences: 1,310,323 | Tokens: 18,280,570 | Elapsed: 42.5 min
28.2% | 300,000,000/1,063,098,071 bytes | Sentences: 1,532,583 | Tokens: 21,964,878 | Elapsed: 54.8 min
32.9% | 349,999,998/1,063,098,071 bytes | Sentences: 1,728,747 | Tokens: 25,611,459 | Elapsed: 92.0 min
37.6% | 399,999,998/1,063,098,071 bytes | Sentences: 1,971,642 | Tokens: 29,311,280 | Elapsed: 105.6 min
42.3% | 450,000,000/1,063,098,071 bytes | Sentences: 2,185,268 | Tokens: 32,965,589 | Elapsed: 117.4 min
47.0% | 499,999,998/1,063,098,071 bytes | Sentences: 2,411,058 | Tokens: 36,

In [7]:

import sqlite3
from pathlib import Path

db_path = Path(r"D:\GPAC\ngram_counts_local.sqlite")

print("Database exists:", db_path.exists())
print("Database size (MB):", round(db_path.stat().st_size / (1024**2), 2))

for suffix in ("-wal", "-shm"):
    p = Path(str(db_path) + suffix)
    print(f"{suffix} exists:", p.exists(),
          f"| size (MB): {round(p.stat().st_size / (1024**2), 2) if p.exists() else 0}")

con = sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True)

print("\nTable counts:")
for table in ("unigrams", "bigrams", "run_state"):
    try:
        print(table, con.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0])
    except sqlite3.Error as e:
        print(table, "ERROR:", e)

print("\nRun status:")
try:
    print(con.execute("""
        SELECT corpus_path, corpus_size, byte_offset,
               sentences, tokens, complete
        FROM run_state
    """).fetchall())
except sqlite3.Error as e:
    print("ERROR:", e)

con.close()


Database exists: True
Database size (MB): 2140.34
-wal exists: False | size (MB): 0
-shm exists: False | size (MB): 0

Table counts:
unigrams 1936354
bigrams 24501063
run_state 1

Run status:
[('D:\\GPAC\\GPAC.txt', 1063098071, 1063098071, 5057775, 77760019, 1)]


In [8]:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path(r"D:\GPAC\ngram_counts_local.sqlite")
con = sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True)

# Total token count from the verified processing run
total_tokens = con.execute(
    "SELECT tokens FROM run_state"
).fetchone()[0]

# Unigram counts and maximum-likelihood probabilities
unigrams = pd.read_sql_query("""
    SELECT token, count
    FROM unigrams
    ORDER BY count DESC
    LIMIT 10
""", con)

unigrams["probability"] = unigrams["count"] / total_tokens

# Bigram counts and conditional probabilities P(second word | first word)
bigrams = pd.read_sql_query("""
    SELECT w1, w2, count
    FROM bigrams
    ORDER BY count DESC
    LIMIT 10
""", con)

# Fetch each first word's total unigram count
first_words = tuple(bigrams["w1"].unique())
placeholders = ",".join("?" for _ in first_words)

word_counts = pd.read_sql_query(
    f"SELECT token, count FROM unigrams WHERE token IN ({placeholders})",
    con,
    params=first_words
)

count_lookup = dict(zip(word_counts["token"], word_counts["count"]))

bigrams["probability"] = bigrams.apply(
    lambda row: row["count"] / count_lookup[row["w1"]],
    axis=1
)

print("Total tokens:", f"{total_tokens:,}")

print("\nTop 10 unigrams:")
display(unigrams)

print("\nTop 10 bigrams:")
display(bigrams)

con.close()


Total tokens: 77,760,019

Top 10 unigrams:


,token,count,probability
0,ነው,1166197,0.014997
1,ላይ,754597,0.009704
2,ውስጥ,346289,0.004453
3,ወደ,330973,0.004256
4,እና,318783,0.004100
5,ጋር,291521,0.003749
6,ነበር,275023,0.003537
7,ግን,271505,0.003492
8,ጊዜ,244085,0.003139
9,ነገር,217662,0.002799



Top 10 bigrams:


,w1,w2,count,probability
0,ዓ,ም,104251,0.970300
1,ነገር,ግን,44743,0.205562
2,ዶ,ር,35618,0.984929
3,ማለት,ነው,32380,0.339823
4,ብቻ,ሳይሆን,32261,0.189372
5,አዲስ,አበባ,31355,0.352922
6,1,1,28958,0.296270
7,ብቻ,ነው,24681,0.144877
8,በአዲስ,አበባ,24036,0.726624
9,ምክር,ቤት,23902,0.558641


In [9]:

from pathlib import Path
import sqlite3
import pandas as pd

# Project paths
PROJECT = Path(r"D:\GPAC")
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

DB = PROJECT / "ngram_counts_local.sqlite"

# Read-only connection: this code will not modify the database
con = sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True)

# Corpus summary
summary = pd.read_sql_query("""
    SELECT corpus_path, corpus_size, sentences, tokens, complete
    FROM run_state
""", con)

summary.to_csv(
    RESULTS / "corpus_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# Top 10 unigrams and their probabilities
top_unigrams = pd.read_sql_query("""
    SELECT token, count,
           CAST(count AS REAL) / (
               SELECT tokens FROM run_state
           ) AS probability
    FROM unigrams
    ORDER BY count DESC
    LIMIT 10
""", con)

top_unigrams.to_csv(
    RESULTS / "top10_unigrams.csv",
    index=False,
    encoding="utf-8-sig"
)

# Top 10 bigrams and conditional probabilities P(w2 | w1)
top_bigrams = pd.read_sql_query("""
    SELECT b.w1, b.w2, b.count,
           CAST(b.count AS REAL) / u.count AS probability
    FROM bigrams AS b
    JOIN unigrams AS u ON u.token = b.w1
    ORDER BY b.count DESC
    LIMIT 10
""", con)

top_bigrams.to_csv(
    RESULTS / "top10_bigrams.csv",
    index=False,
    encoding="utf-8-sig"
)

con.close()

print("Saved files:")
for path in sorted(RESULTS.glob("*.csv")):
    print(path.name, "|", path.stat().st_size, "bytes")

print("\nTop unigrams:")
display(top_unigrams)

print("\nTop bigrams:")
display(top_bigrams)


Saved files:
corpus_summary.csv | 102 bytes
top10_bigrams.csv | 441 bytes
top10_unigrams.csv | 401 bytes

Top unigrams:


,token,count,probability
0,ነው,1166197,0.014997
1,ላይ,754597,0.009704
2,ውስጥ,346289,0.004453
3,ወደ,330973,0.004256
4,እና,318783,0.004100
5,ጋር,291521,0.003749
6,ነበር,275023,0.003537
7,ግን,271505,0.003492
8,ጊዜ,244085,0.003139
9,ነገር,217662,0.002799



Top bigrams:


,w1,w2,count,probability
0,ዓ,ም,104251,0.970300
1,ነገር,ግን,44743,0.205562
2,ዶ,ር,35618,0.984929
3,ማለት,ነው,32380,0.339823
4,ብቻ,ሳይሆን,32261,0.189372
5,አዲስ,አበባ,31355,0.352922
6,1,1,28958,0.296270
7,ብቻ,ነው,24681,0.144877
8,በአዲስ,አበባ,24036,0.726624
9,ምክር,ቤት,23902,0.558641


In [10]:

import sqlite3
from pathlib import Path
import pandas as pd

db_path = Path(r"D:\GPAC\ngram_counts_local.sqlite")
con = sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True)

# Inspect the available tables
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", con)

print("Tables in database:")
display(tables)

# Correct conditional probability:
# P(w2 | w1) = count(w1, w2) / total counts of all bigrams starting with w1
top_bigrams_corrected = pd.read_sql_query("""
    WITH outgoing AS (
        SELECT w1, SUM(count) AS outgoing_count
        FROM bigrams
        GROUP BY w1
    )
    SELECT b.w1, b.w2, b.count,
           o.outgoing_count,
           CAST(b.count AS REAL) / o.outgoing_count AS probability
    FROM bigrams AS b
    JOIN outgoing AS o ON b.w1 = o.w1
    ORDER BY b.count DESC
    LIMIT 10
""", con)

print("Top 10 bigrams with corrected conditional probabilities:")
display(top_bigrams_corrected)

con.close()


Tables in database:


,name
0,bigrams
1,run_state
2,unigrams


Top 10 bigrams with corrected conditional probabilities:


,w1,w2,count,outgoing_count,probability
0,ዓ,ም,104251,107410,0.970589
1,ነገር,ግን,44743,216514,0.206652
2,ዶ,ር,35618,36155,0.985147
3,ማለት,ነው,32380,94568,0.342399
4,ብቻ,ሳይሆን,32261,169139,0.190737
5,አዲስ,አበባ,31355,88829,0.352982
6,1,1,28958,97378,0.297377
7,ብቻ,ነው,24681,169139,0.145921
8,በአዲስ,አበባ,24036,33079,0.726624
9,ምክር,ቤት,23902,42750,0.559111


In [11]:

import sqlite3
import pandas as pd
from pathlib import Path

# File locations
db_path = Path(r"D:\GPAC\ngram_counts_local.sqlite")
results_dir = Path(r"D:\GPAC\results")
output_path = results_dir / "top10_bigrams.csv"

results_dir.mkdir(parents=True, exist_ok=True)

# Calculate and export the corrected top 10 bigrams
query = """
WITH top_bigrams AS (
    SELECT w1, w2, count
    FROM bigrams
    ORDER BY count DESC
    LIMIT 10
),
outgoing AS (
    SELECT b.w1, SUM(b.count) AS outgoing_count
    FROM bigrams AS b
    JOIN (
        SELECT DISTINCT w1
        FROM top_bigrams
    ) AS t ON b.w1 = t.w1
    GROUP BY b.w1
)
SELECT
    t.w1,
    t.w2,
    t.count,
    o.outgoing_count,
    ROUND(
        CAST(t.count AS REAL) / o.outgoing_count,
        6
    ) AS probability
FROM top_bigrams AS t
JOIN outgoing AS o ON t.w1 = o.w1
ORDER BY t.count DESC;
"""

with sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True) as conn:
    df = pd.read_sql_query(query, conn)

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Export complete!")
print(f"Saved to: {output_path}")
print(f"Rows exported: {len(df)}")
display(df)


Export complete!
Saved to: D:\GPAC\results\top10_bigrams.csv
Rows exported: 10


,w1,w2,count,outgoing_count,probability
0,ዓ,ም,104251,107410,0.970589
1,ነገር,ግን,44743,216514,0.206652
2,ዶ,ር,35618,36155,0.985147
3,ማለት,ነው,32380,94568,0.342399
4,ብቻ,ሳይሆን,32261,169139,0.190737
5,አዲስ,አበባ,31355,88829,0.352982
6,1,1,28958,97378,0.297377
7,ብቻ,ነው,24681,169139,0.145921
8,በአዲስ,አበባ,24036,33079,0.726624
9,ምክር,ቤት,23902,42750,0.559111


In [13]:

import sys
import subprocess

print("Python location:", sys.executable)

subprocess.check_call([
    sys.executable, "-m", "ensurepip", "--upgrade"
])


Python location: C:\Users\Beam\AppData\Local\Programs\Python\Python314\python.exe


0

In [15]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import streamlit as st
print("Streamlit version:", st.__version__)

Streamlit version: 1.66.1
